In [16]:
'''
ÖLÜM VERİLERİ
total_deaths / new_deaths / new_deaths_smoothed	Aynı mantık, ölüm için
*_per_million uzantılı olanlar	Yukarıdakilerin milyon kişi başına normalize edilmiş hali — 
ülkeler arası karşılaştırma için bunları kullanacağız
'''

'\nÖLÜM VERİLERİ\ntotal_deaths / new_deaths / new_deaths_smoothed\tAynı mantık, ölüm için\n*_per_million uzantılı olanlar\tYukarıdakilerin milyon kişi başına normalize edilmiş hali — \nülkeler arası karşılaştırma için bunları kullanacağız\n'

In [17]:
'''
Aşırı Ölüm (Excess Mortality)
excess_mortality	Beklenen ölüm sayısına göre fazladan ölüm yüzdesi
(COVID'in resmi sayılara yansımayan gerçek etkisini gösterir)
excess_mortality_cumulative	Kümülatif hali
excess_mortality_cumulative_absolute	Mutlak sayı olarak
excess_mortality_cumulative_per_million	Milyon kişi başına 
'''

"\nAşırı Ölüm (Excess Mortality)\nexcess_mortality\tBeklenen ölüm sayısına göre fazladan ölüm yüzdesi\n(COVID'in resmi sayılara yansımayan gerçek etkisini gösterir)\nexcess_mortality_cumulative\tKümülatif hali\nexcess_mortality_cumulative_absolute\tMutlak sayı olarak\nexcess_mortality_cumulative_per_million\tMilyon kişi başına \n"

In [18]:
#kullancağımız kütüphaneleri yüklüyoruz
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#weekly_cases.csv dosyasını kullanacağız
df_countries = pd.read_csv(
    r'C:\Users\nazlican\OneDrive\Masaüstü\CoronaVirusProject\data\processed\countries_clean.csv',
    parse_dates=['date']
)

print("Satır-sütun sayısı:", df_countries.shape)
print("Ülke sayısı:", df_countries['location'].nunique())
print("İlk tarih:", df_countries['date'].min())
print("Son tarih:", df_countries['date'].max())

print(
    "Kalan ülke-tarih tekrarı:",
    df_countries.duplicated(
        subset=['location', 'date']
    ).sum()
)

Satır-sütun sayısı: (324819, 67)
Ülke sayısı: 194
İlk tarih: 2020-01-01 00:00:00
Son tarih: 2024-08-14 00:00:00
Kalan ülke-tarih tekrarı: 0


In [19]:
#total_deaths sütunu için toplam kaç dolu kaç boş satır olduğunu tespit ediyoruz
death_coverage = df_countries.groupby('location').agg(
    toplam_satir=('total_deaths', 'size'),
    dolu_olum_kaydi=('total_deaths', 'count')
)

death_coverage['bos_olum_kaydi'] = (
    death_coverage['toplam_satir']
    - death_coverage['dolu_olum_kaydi']
)

print(
    death_coverage
    .sort_values('dolu_olum_kaydi')
    .head(15)
)

                     toplam_satir  dolu_olum_kaydi  bos_olum_kaydi
location                                                          
Afghanistan                  1674             1674               0
Albania                      1674             1674               0
Algeria                      1674             1674               0
Andorra                      1674             1674               0
Angola                       1674             1674               0
Antigua and Barbuda          1674             1674               0
Argentina                    1678             1674               4
Armenia                      1674             1674               0
Australia                    1674             1674               0
Austria                      1674             1674               0
Azerbaijan                   1674             1674               0
Bahamas                      1674             1674               0
Bahrain                      1674             1674            

In [20]:
#hem dolu hem boş olan ölüm kayıtlarına sahip ülkeler
partial_death_missing = death_coverage[
    (death_coverage['dolu_olum_kaydi'] > 0)
    & (death_coverage['bos_olum_kaydi'] > 0)
].copy()

#sadece eksik ölüm kayıtlarını seçiyoruz
missing_death_rows = df_countries.loc[
    df_countries['location'].isin(partial_death_missing.index)
    & df_countries['total_deaths'].isna(),
    ['location', 'date', 'total_deaths']
].copy()

#sadece dolu ölüm kayıtlarını seçiyoruz
valid_death_range = (
    df_countries.loc[
        df_countries['total_deaths'].notna()
    ]
    .groupby('location')
    .agg(
        ilk_dolu_olum_tarihi=('date', 'min'),
        son_dolu_olum_tarihi=('date', 'max')
    )
)

missing_death_positions = missing_death_rows.join(
    valid_death_range,
    on='location'
)

missing_death_positions['bosluk_konumu'] = 'arada'

missing_death_positions.loc[
    missing_death_positions['date']
    < missing_death_positions['ilk_dolu_olum_tarihi'],
    'bosluk_konumu'
] = 'başta'

missing_death_positions.loc[
    missing_death_positions['date']
    > missing_death_positions['son_dolu_olum_tarihi'],
    'bosluk_konumu'
] = 'sonda'

death_missing_position_summary = (
    missing_death_positions
    .groupby(['location', 'bosluk_konumu'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['başta', 'arada', 'sonda'], fill_value=0)
)

print(death_missing_position_summary)

bosluk_konumu  başta  arada  sonda
location                          
Argentina          4      0      0
Czechia            0      0      8
Estonia            0      0      8
India              0      0      8
Italy              0      0      3
Lithuania          0      0     10
Malaysia           0      0     10
Mexico             4      0      0
New Zealand        0      0      7
Thailand           1      0      0


In [21]:
#toplam ölüm kayıtlarının negatif azaldığı yerleri tespit ediyoruz
death_changes = (
    df_countries[
        ['location', 'date', 'total_deaths']
    ]
    .sort_values(['location', 'date'])
    .copy()
)

death_changes['degisim'] = (
    death_changes
    .groupby('location')['total_deaths']
    .diff()
)

death_decreases = death_changes.loc[
    death_changes['degisim'] < 0
].copy()

print("Toplam ölüm sayısının azaldığı kayıt sayısı:", len(death_decreases))
print(death_decreases.to_string(index=False))

Toplam ölüm sayısının azaldığı kayıt sayısı: 9
        location       date  total_deaths  degisim
       Australia 2023-07-23       22694.0    -76.0
          Canada 2022-06-19       41017.0   -331.0
           Chile 2023-04-02       61050.0  -3432.0
           China 2024-01-14      121916.0    -22.0
       Indonesia 2024-02-11      162050.0     -4.0
          Panama 2023-05-07        8620.0     -1.0
Papua New Guinea 2023-08-13         640.0    -30.0
    Sierra Leone 2022-06-12         125.0     -1.0
        Thailand 2020-11-22           0.0    -60.0


In [22]:
#eksi olarak azalan yerlerin neyden dolayı kaynaklandığını tespit etmeye çalışıyoruz
death_decrease_details = death_decreases.merge(
    df_countries[
        ['location', 'date', 'new_deaths']
    ],
    on=['location', 'date'],
    how='left',#soldaki değerleri korumak için 
    validate='one_to_one'
)

print(death_decrease_details.to_string(index=False))

        location       date  total_deaths  degisim  new_deaths
       Australia 2023-07-23       22694.0    -76.0         NaN
          Canada 2022-06-19       41017.0   -331.0         NaN
           Chile 2023-04-02       61050.0  -3432.0         NaN
           China 2024-01-14      121916.0    -22.0         NaN
       Indonesia 2024-02-11      162050.0     -4.0         NaN
          Panama 2023-05-07        8620.0     -1.0         NaN
Papua New Guinea 2023-08-13         640.0    -30.0         NaN
    Sierra Leone 2022-06-12         125.0     -1.0         NaN
        Thailand 2020-11-22           0.0    -60.0         NaN


In [24]:
#new_deaths sütununda toplam ölüm kaydı,boş ölüm kaydı incelenmesi
new_death_coverage = df_countries.groupby('location').agg(
    toplam_satir=('new_deaths', 'size'),
    dolu_yeni_olum_kaydi=('new_deaths', 'count')
)

new_death_coverage['bos_yeni_olum_kaydi'] = (
    new_death_coverage['toplam_satir']
    - new_death_coverage['dolu_yeni_olum_kaydi']
)

print(
    new_death_coverage
    .sort_values('bos_yeni_olum_kaydi', ascending=False)
    .head(15)
)

             toplam_satir  dolu_yeni_olum_kaydi  bos_yeni_olum_kaydi
location                                                            
France               1674                  1274                  400
Germany              1674                  1282                  392
Spain                1674                  1282                  392
Malaysia             1684                  1674                   10
Lithuania            1684                  1674                   10
Estonia              1682                  1674                    8
India                1682                  1674                    8
Czechia              1682                  1674                    8
New Zealand          1681                  1674                    7
Argentina            1678                  1674                    4
Mexico               1678                  1674                    4
Italy                1677                  1674                    3
Thailand             1675         

In [25]:
#total_deaths de olduğu gibi eksik ölümler veri setinin başında mı ortasında mı sonunda mı tespit ediyoruz
partial_new_death_missing = new_death_coverage[
    (new_death_coverage['dolu_yeni_olum_kaydi'] > 0)
    & (new_death_coverage['bos_yeni_olum_kaydi'] > 0)
].copy()

missing_new_death_rows = df_countries.loc[
    df_countries['location'].isin(
        partial_new_death_missing.index
    )
    & df_countries['new_deaths'].isna(),
    ['location', 'date', 'new_deaths']
].copy()

valid_new_death_range = (
    df_countries.loc[
        df_countries['new_deaths'].notna()
    ]
    .groupby('location')
    .agg(
        ilk_dolu_yeni_olum_tarihi=('date', 'min'),
        son_dolu_yeni_olum_tarihi=('date', 'max')
    )
)

missing_new_death_positions = missing_new_death_rows.join(
    valid_new_death_range,
    on='location'
)

missing_new_death_positions['bosluk_konumu'] = 'arada'

missing_new_death_positions.loc[
    missing_new_death_positions['date']
    < missing_new_death_positions[
        'ilk_dolu_yeni_olum_tarihi'
    ],
    'bosluk_konumu'
] = 'başta'

missing_new_death_positions.loc[
    missing_new_death_positions['date']
    > missing_new_death_positions[
        'son_dolu_yeni_olum_tarihi'
    ],
    'bosluk_konumu'
] = 'sonda'

new_death_missing_position_summary = (
    missing_new_death_positions
    .groupby(['location', 'bosluk_konumu'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['başta', 'arada', 'sonda'], fill_value=0)
)

print(new_death_missing_position_summary)

bosluk_konumu     başta  arada  sonda
location                             
Argentina             4      0      0
Australia             0      1      0
Canada                0      1      0
Chile                 0      1      0
China                 0      1      0
Czechia               0      0      8
Estonia               0      0      8
France                0      0    400
Germany               0      0    392
India                 0      0      8
Indonesia             0      1      0
Italy                 0      0      3
Lithuania             0      0     10
Malaysia              0      0     10
Mexico                4      0      0
New Zealand           0      0      7
Panama                0      1      0
Papua New Guinea      0      1      0
Sierra Leone          0      1      0
Spain                 0      0    392
Thailand              1      1      0


In [27]:
#negatif günlük ölüm kayıtlarını kontrol etme,hatalı giriş var mı kontrol ediyoruz
new_death_changes = (
    df_countries[
        ['location', 'date', 'new_deaths']
    ]
    .sort_values(['location', 'date'])
    .copy()
)

negative_new_deaths = new_death_changes.loc[
    new_death_changes['new_deaths'] < 0
].copy()

print(
    "Negatif new_deaths kayıt sayısı:",
    len(negative_new_deaths)
)

print(negative_new_deaths.to_string(index=False))

Negatif new_deaths kayıt sayısı: 0
Empty DataFrame
Columns: [location, date, new_deaths]
Index: []
